# AutoGen

**Module:** 11-agent-frameworks

**Notebook:** `04-autogen.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **AutoGen Overview** with clear contracts and failure modes
- Explain and apply **Two-Agent Chat** with clear contracts and failure modes
- Explain and apply **Group Chat** with clear contracts and failure modes
- Explain and apply **Termination Conditions** with clear contracts and failure modes
- Explain and apply **Code Execution Caution** with clear contracts and failure modes
- Explain and apply **Fit** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — AutoGen

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **AutoGen Overview**
2. **Two-Agent Chat**
3. **Group Chat**
4. **Termination Conditions**
5. **Code Execution Caution**
6. **Fit**

Read top-to-bottom once, then revisit weak spots with the exercises.


## AutoGen Overview

### Definition
**AutoGen Overview** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around AutoGen Overview typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For AutoGen Overview: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain AutoGen Overview as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating AutoGen Overview as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for AutoGen Overview
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use AutoGen Overview when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does AutoGen Overview improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "AutoGen Overview" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "AutoGen Overview"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


In [ ]:
# Demo: decision table for applying "AutoGen Overview"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_autogen_over", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Two-Agent Chat

### Definition
**Two-Agent Chat** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Two-Agent Chat typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Two-Agent Chat: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Two-Agent Chat as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Two-Agent Chat as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Two-Agent Chat
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Two-Agent Chat when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Two-Agent Chat" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Two-Agent Chat"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


### Worked scenario — Two-Agent Chat

**Situation:** A team wants to productionize a feature involving **Two-Agent Chat**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Group Chat

### Definition
**Group Chat** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Group Chat typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Group Chat: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Group Chat as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Group Chat as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Group Chat
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Group Chat when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Group Chat" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Group Chat"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Group Chat"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Group Chat"}
strong = {"definition": "Group Chat", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Group Chat"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Group Chat", "passed": len(checks)-len(failed), "failed": failed})


## Termination Conditions

### Definition
**Termination Conditions** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Termination Conditions typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Termination Conditions: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Termination Conditions as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Termination Conditions as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Termination Conditions
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Termination Conditions when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Termination Conditions" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Termination Conditions"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Termination Conditions"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Termination Conditions"}
strong = {"definition": "Termination Conditions", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Termination Conditions"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Termination Conditions", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Termination Conditions

**Situation:** A team wants to productionize a feature involving **Termination Conditions**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Code Execution Caution

### Definition
**Code Execution Caution** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Code Execution Caution typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Code Execution Caution: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Code Execution Caution as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Code Execution Caution as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Code Execution Caution
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Code Execution Caution when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Code Execution Caution" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Code Execution Caution"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Code Execution Caution"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Code Execution Caution"}
strong = {"definition": "Code Execution Caution", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Code Execution Caution"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Code Execution Caution", "passed": len(checks)-len(failed), "failed": failed})


## Fit

### Definition
**Fit** is a core building block in 04-autogen within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Fit typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Fit: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Fit as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Fit as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Fit
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Fit when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Fit" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Fit"
    notebook: str = "04-autogen"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Fit"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Fit"}
strong = {"definition": "Fit", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Fit"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Fit", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Fit

**Situation:** A team wants to productionize a feature involving **Fit**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **AutoGen**.

| Topic | Do | Don't |
|-------|----|-------|
| AutoGen Overview | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Two-Agent Chat | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Group Chat | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Termination Conditions | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Code Execution Caution | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Fit | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| AutoGen Overview | Key concept covered in this notebook; see its section for definition and pitfalls |
| Two-Agent Chat | Key concept covered in this notebook; see its section for definition and pitfalls |
| Group Chat | Key concept covered in this notebook; see its section for definition and pitfalls |
| Termination Conditions | Key concept covered in this notebook; see its section for definition and pitfalls |
| Code Execution Caution | Key concept covered in this notebook; see its section for definition and pitfalls |
| Fit | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **AutoGen** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **11-agent-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **AutoGen Overview**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Two-Agent Chat**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Group Chat**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Termination Conditions**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Code Execution Caution**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
